# 🏔️ Transformer-to-Coords

Explore the JAX-native Transformer architecture that maps amino acid sequences directly to 3D Cartesian coordinates.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/resonance-flow/blob/main/examples/interactive_tutorials/transformer_to_coords.ipynb)

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/elkins/resonance-flow.git git+https://github.com/elkins/diff-biophys.git biotite matplotlib
else:
    sys.path.append("../../")

import matplotlib.pyplot as plt
import numpy as np

## 1. Initializing the Transformer Predictor

ResonanceFlow uses an autoregressive or one-shot Transformer that takes in sequence and NMR embeddings to hallucinate 3D coordinates.

In [ ]:
import jax

from resonance_flow.model import TransformerCoordinatePredictor

model = TransformerCoordinatePredictor(d_model=64, num_layers=2)
print(f"Model initialized: {model}")

key = jax.random.PRNGKey(42)
L = 50
dummy_seq = jax.random.randint(key, (1, L), 0, 20)

variables = model.init(key, dummy_seq)
predicted_coords = model.apply(variables, dummy_seq)
coords = np.array(predicted_coords[0])
print(f"\nOutput coordinates shape: {coords.shape}")

## 2. Converting Tensors to Physical Structures

The model output is just a raw `[L, 3]` tensor of C-alpha coordinates. To visualize it, we can use `biotite` to build a minimal PDB file.

In [ ]:
import biotite.structure as struc
import biotite.structure.io.pdb as pdb

scaled_coords = coords * 5.0

array = struc.AtomArray(L)
array.coord = scaled_coords
array.atom_name = ["CA"] * L
array.element = ["C"] * L
array.res_id = np.arange(1, L + 1)
array.res_name = ["GLY"] * L
array.chain_id = ["A"] * L

pdb_file = pdb.PDBFile()
pdb.set_structure(pdb_file, array)
pdb_file.write("predicted_trace.pdb")
print("Saved predicted C-alpha trace to predicted_trace.pdb")

## 3. Visualizing the Transformer Output

Let's look at what the untrained Transformer generated. Because the weights are randomly initialized, it will look like a random coil or a collapsed globule. During training, the gradients from `rdc_loss` and biophysical penalties will sculpt this cloud into a folded protein!

In [ ]:
import py3Dmol

with open("predicted_trace.pdb") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=400)
view.addModel(pdb_data, "pdb")
view.setStyle({"model": -1}, {"line": {"color": "spectrum", "linewidth": 5}})
view.zoomTo()
view.show()

## 4. Static 3D Plot (Fallback)

In environments where the interactive `py3Dmol` widget cannot be rendered (e.g., static HTML exports, GitHub previews), here is a static Matplotlib plot of the predicted trace.

In [ ]:
import numpy as np

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
c = np.array(coords) * 5.0  # scaled coords
ax.plot(c[:, 0], c[:, 1], c[:, 2], marker="o", linestyle="-", color="b", markersize=4)
ax.set_title("Predicted C-alpha Trace (Static)")
ax.set_xlabel("X (Å)")
ax.set_ylabel("Y (Å)")
ax.set_zlabel("Z (Å)")
plt.show()